# Results of a run

The notebook only reads the logs of a finished (or interrupted) run from `results/<RUN_NAME>/`.
To produce them, launch `python -m src.run --name <RUN_NAME>` from the project root (see `README.md`).

In [ ]:
import json
import sys

sys.path.insert(0, "..")

import pandas as pd
from IPython.display import Image, display

from src.config import load_config, resolve

RUN_NAME = "main"

run_dir = resolve(load_config()["results_dir"]) / RUN_NAME
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
print("Detector:", summary["detector_model"], "| attacker:", summary["attacker_model"])
print("Function pairs:", summary["pairs"], "| max rounds:", summary["max_rounds"])

## 1. Quality without attack

In [ ]:
pd.DataFrame({name: c["baseline"] for name, c in summary["configs"].items()}).T.rename_axis("configuration")

## 2. Attack success rate by round budget

In [ ]:
curve = pd.DataFrame(
    [{"configuration": name, **row} for name, c in summary["configs"].items() for row in c["asr_curve"]]
)
display(curve.pivot(index="configuration", columns="R", values="asr"))
display(curve)
display(Image(filename=str(run_dir / "asr_curve.png")))

In [ ]:
pd.DataFrame(
    {
        name: {
            "control flip rate": c["control"]["flip_rate"],
            "control flips": c["control"]["flips"],
            "control trials": c["control"]["trials"],
            "invalid answers": sum(c["invalid_answers"].values()),
            "mean rounds to success": c["mean_rounds_to_success"],
            "successful styles": c["successful_styles"],
            "attacks stopped by comment filter": c["attacks_stopped_by_comment_filter"],
        }
        for name, c in summary["configs"].items()
    }
).T.rename_axis("configuration")

## 3. Rounds of one attack

In [ ]:
CONFIGURATION = "B"

rounds = pd.read_json(run_dir / "rounds.jsonl", lines=True)
rounds = rounds[rounds["config"] == CONFIGURATION]
if len(rounds):
    one = rounds[rounds["id"] == rounds["id"].iloc[0]]
    print("Function:", one["id"].iloc[0])
    for _, r in one.iterrows():
        print(f"--- round {r['round']} | style {r.get('style')} | verdict {r.get('verdict')}")
        print(r.get("comment"))
        print("Explanation:", r.get("explanation"))
        print()
else:
    print("No attack rounds for this configuration.")